In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import Markdown, display
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score, roc_curve,
)

project_dir = Path(r"D:\credit")
feature_dir = project_dir / "data" / "xgboost_features"
result_dir = project_dir / "results" / "XGBOOST"
result_dir.mkdir(parents=True, exist_ok=True)

random_state = 42
threshold = 0.5

In [ ]:
metadata = json.loads(
    (feature_dir / "metadata.json").read_text(encoding="utf-8")
)
target_column = metadata["target"]
selected_features = metadata["feature_columns"]

def load_dataset(split_name):
    data = pd.read_parquet(feature_dir / f"{split_name}.parquet")
    features = data.drop(columns=target_column)
    target = data[target_column]
    if features.columns.tolist() != selected_features:
        raise ValueError(f"{split_name} 的特征名称或顺序与保存记录不一致")
    return features, target

X_train, y_train = load_dataset("train")
X_val, y_val = load_dataset("val")
X_test, y_test = load_dataset("test")

print(f"训练集：{X_train.shape}，验证集：{X_val.shape}，测试集：{X_test.shape}")

## 1. 模型训练

1. 使用直方图算法，树深度 5、学习率 0.05，最多训练 1,000 轮；
2. 验证集 AUC 连续 50 轮未提升时停止。预测和特征重要性使用最佳轮次对应的模型。
3. 日志中的 train-auc、val-auc 分别为训练集、验证集 AUC；下方说明最佳轮数及其验证集 AUC。

In [ ]:
train_matrix = xgb.DMatrix(X_train, label=y_train)
val_matrix = xgb.DMatrix(X_val, label=y_val)
test_matrix = xgb.DMatrix(X_test, label=y_test)
params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "max_depth": 5,
    "min_child_weight": 5,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "seed": random_state,
    "n_jobs": -1,
}
history = {}
model = xgb.train(
    params, train_matrix, num_boost_round=1000,
    evals=[(train_matrix, "train"), (val_matrix, "val")],
    early_stopping_rounds=50, evals_result=history, verbose_eval=100,
)
best_rounds = model.best_iteration + 1
best_val_auc = float(model.best_score)
model = model[:best_rounds]
display(Markdown(
    f"**训练结果：** 最佳迭代轮数 **{best_rounds}**，对应验证集 AUC **{best_val_auc:.4f}**。"
))

##  2. 主要评估结果

| 指标 | 含义 |
| --- | --- |
| AUC | 违约概率的排序能力，越高表示两类样本越容易区分 |
| KS | ROC 曲线上 TPR 与 FPR 的最大差值，越高表示两类样本区分度越大 |
| Precision | 被预测为违约的样本中，实际违约的比例 |
| Recall | 实际违约样本中，被模型识别的比例 |
| F1 | Precision 与 Recall 的调和平均值 |


In [ ]:
def markdown_table(frame):
    table = frame.reset_index().round(4)
    header = "| " + " | ".join(map(str, table.columns)) + " |"
    separator = "| " + " | ".join(["---"] * len(table.columns)) + " |"
    rows = ["| " + " | ".join(map(str, row)) + " |" for row in table.itertuples(index=False, name=None)]
    return "\n".join([header, separator, *rows])

In [ ]:
probabilities = {
    "训练集": model.predict(train_matrix),
    "验证集": model.predict(val_matrix),
    "测试集": model.predict(test_matrix),
}
labels = {"训练集": y_train, "验证集": y_val, "测试集": y_test}
metric_rows = []
for name, target in labels.items():
    probability = probabilities[name]
    prediction = (probability >= threshold).astype(int)
    false_positive_rate, true_positive_rate, _ = roc_curve(target, probability)
    metric_rows.append({
        "样本": name,
        "AUC": roc_auc_score(target, probability),
        "KS": (true_positive_rate - false_positive_rate).max(),
        "Precision": precision_score(target, prediction),
        "Recall": recall_score(target, prediction),
        "F1": f1_score(target, prediction),
    })
metrics = pd.DataFrame(metric_rows).set_index("样本")
test_probability = probabilities["测试集"]
test_prediction = (test_probability >= threshold).astype(int)
test_metrics = metrics.loc["测试集"]
confusion = pd.DataFrame(
    confusion_matrix(y_test, test_prediction, labels=[0, 1]),
    index=["实际正常", "实际违约"], columns=["预测正常", "预测违约"],
).rename_axis("真实标签")

display(Markdown("### 指标汇总\n\n" + markdown_table(metrics)))
display(Markdown("### 测试集混淆矩阵\n\n" + markdown_table(confusion)))
result_text = (
    f"测试集 AUC 为 **{test_metrics['AUC']:.4f}**，KS 为 **{test_metrics['KS']:.4f}**。"
    f"阈值为 {threshold} 时，模型识别出实际违约样本的 **{test_metrics['Recall']:.2%}**；"
    f"被预测为违约的样本中，实际违约比例为 **{test_metrics['Precision']:.2%}**，"
    f"F1 为 **{test_metrics['F1']:.4f}**。"
)
display(Markdown("**结果解读：** " + result_text))

## 3. 学习曲线与 ROC 曲线

左图展示训练、验证 AUC 随迭代轮次的变化，虚线标记最佳轮次，用于观察训练过程。
右图为测试集 ROC：横轴是正常样本误报率，纵轴是违约样本召回率；
越靠近左上角，表示相同误报率下能够识别更多违约样本。

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train"]["auc"], label="Train")
axes[0].plot(history["val"]["auc"], label="Validation")
axes[0].axvline(best_rounds - 1, color="grey", linestyle="--", label="Best round")
axes[0].set(xlabel="Boosting iteration (0-based)", ylabel="AUC", title="Learning curve")
axes[0].legend()
false_positive_rate, true_positive_rate, _ = roc_curve(y_test, test_probability)
axes[1].plot(false_positive_rate, true_positive_rate, label=f"Test AUC = {test_metrics['AUC']:.4f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="grey")
axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title="Test ROC")
axes[1].legend()
figure.tight_layout()
plt.show()

## 4. 特征重要性

使用最佳轮次模型的 gain（特征参与分裂时的平均增益）排序，展示前 20 个特征。
数值越大，表示该特征参与的分裂平均带来的目标函数改善越大。

In [ ]:
importance = (
    pd.Series(model.get_score(importance_type="gain"), name="gain")
    .reindex(X_train.columns, fill_value=0)
    .sort_values(ascending=False)
    .rename_axis("feature")
    .to_frame()
)
top_features = importance.head(20)
top_feature_text = "、".join(top_features.index[:5])
display(Markdown(markdown_table(top_features)))
display(Markdown(f"**排序结果：** 按平均分裂增益，前五个特征为 {top_feature_text}。"))
figure, axis = plt.subplots(figsize=(9, 7))
top_features.sort_values("gain").plot.barh(ax=axis, legend=False)
axis.set(xlabel="Average gain", ylabel="", title="Top 20 feature importance")
figure.tight_layout()
plt.show()

## 5. 导出三模型比较指标

使用上方本次运行的指标导出 results/XGBOOST/metrics.csv，不重新训练，不生成分析报告。
比较前请先运行三个模型并分别导出新指标，再运行 code/model/ModelComparison.ipynb；已有旧 CSV 不代表本次结果。

In [ ]:
export_metrics = metrics.assign(
    样本数=[len(y_train), len(y_val), len(y_test)],
    特征数=len(selected_features), 阈值=threshold,
    结果来源="当前 Parquet 特征，本次运行",
).reset_index()
export_metrics.to_csv(result_dir / "metrics.csv", index=False, encoding="utf-8-sig")
display(export_metrics)